# Sentinel-1 Fast Ice Detection using Local GeoTIFF Files

## Data sources

- A local image pair separated by a defined temporal baseline (e.g. 12 days) for a region of interest (ROI). Images must be in dB.
- Sample images are provided and can be downloaded for testing.

## Pipeline overview

1. **Define or download image pair** — specify local images, or download the provided samples.
2. **Check and trim** — verify spatial overlap and temporal baseline, then reproject and trim the pair to their common valid extent.
3. **Compute NormCovar** — calculate the normalised covariance between the image pair. NormCovar is a windowed, normalised measure of how strongly two SAR images vary together at each pixel, computed by dividing local covariance by the square root of local variance. Output is an upsampled **RGB** image, with an optional landmask applied.
4. **SAM - Setup** — initialise the Segment Anything Model, using GPU acceleration if available.
5. **SAM - Segment** — pass the RGB image to SAM to generate image segments.
6. **SVM - Setup** - Set up a Support Vector Machine (SVM) classifier. Retrieve and train if weights don't exist.
7. **SVM - Classify Image segments** - Perform binary classification of image segments as **fast ice** / not fast ice.
8. **Results - Write to file** - Save the prediction to a geotiff.

## 1. Imports

In [ ]:
import pathlib
import sys
import urllib.request
from pathlib import Path

import numpy as np
from osgeo import gdal
from normalized_covariance import normcovar, normcovar_utils
import re


## 2. Parameters

Set up file pathing

In [ ]:
DATA_DIR = pathlib.Path("./data")
site = "test"

GEOTIFF_DIR = DATA_DIR / site
GEOTIFF_DIR.mkdir(parents=True, exist_ok=True)

test_scene_1_s3_path = "https://data.dev.dea.ga.gov.au/experimental/baseline/mcmurdo/ga_s1_nrb_ew_hh_hv_1/2019/12/17/S1A_EW_GRDM_1SDH_20191217T112553_20191217T112658_030387_037A1C_58E9/ga_s1a_nrb_1-0-0__EW___A_20191217T112553_HH-gamma0_db.tif"
test_scene_2_s3_path = "https://data.dev.dea.ga.gov.au/experimental/baseline/mcmurdo/ga_s1_nrb_ew_hh_hv_1/2019/12/29/S1A_EW_GRDM_1SDH_20191229T112553_20191229T112657_030562_038020_E608/ga_s1a_nrb_1-0-0__EW___A_20191229T112553_HH-gamma0_db.tif"

for url in [test_scene_1_s3_path,test_scene_2_s3_path]:
    filename = url.split("/")[-1]
    dest = GEOTIFF_DIR / filename
    if not dest.exists():
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(url, dest)
        print(f"Saved to {dest}")

## 3. Load and prepare data

`normcovar_utils.check_and_trim_image_pair`

Manually set date and supply to check and trim function

In [ ]:
# Test pair: S1_image_pair_20210708T145617_20210720T145618
image_1 = GEOTIFF_DIR / Path(test_scene_1_s3_path).name
image_2 = GEOTIFF_DIR / Path(test_scene_2_s3_path).name 
img_pair = [image_1, image_2]

date1 = re.search(r"(\d{8}T\d{6})", str(Path(test_scene_1_s3_path).name)).group(1)
date2 = re.search(r"(\d{8}T\d{6})", str(Path(test_scene_2_s3_path).name)).group(1)

# Define output directory
IMG_PAIR_DIR = GEOTIFF_DIR / f"S1_image_pair_{date1}_{date2}"
print(image_1)
print(image_2)
print(IMG_PAIR_DIR)

In [ ]:
# Temporal baseline in days
min_temp_baseline = 11.9
max_temp_baseline = 12.1

# Output epsg
output_epsg = 3031

In [ ]:
normcovar_utils.check_and_trim_image_pair(
    img_pair,
    IMG_PAIR_DIR,
    min_temp_baseline = min_temp_baseline,
    max_temp_baseline = max_temp_baseline,
    output_epsg = output_epsg,
    date1 = date1,
    date2 = date2,
    overwrite = False,
)


## 4. Process image pair

A single call to `normcovar.fully_process_single_image_pair` computes the Normalised Covariance between the images and writes the results to GeoTIFF files in `IMG_PAIR_DIR`, including the windowed normcovar bands, the false-colour RGB composite (`normcovar__RGB.tif`), a resampled RGB (`normcovar__RGB__resampled_<n>_<n>.tif`), and — if a landmask shapefile is supplied — a matching landmask (`landmask__resampled_<n>_<n>.tif`).

The resampled RGB, with the landmask applied, should be used as input to the SAM model.

The cell below will fetch and unzip the SCAR ADD medium-res coastline shapefile.

In [ ]:
# Define window sizes to process
window_list = [11,21,33]

# Save intermediate normprod steps?
save_intermediate_products = True

# Define min/max values for normprod scaling to RGB image
NP_min = -0.5
NP_max = 1.0

# Resample NP RGB image
resample = True

# Set resamping interval for NP RGB image and landmask
resample_interval = 10

In [ ]:
from fast_ice.datasets.coastline import fetch_add_coastline_shapefile
landmask_shapefile_path = fetch_add_coastline_shapefile(output_dir='data')

In [ ]:
normcovar.fully_process_single_image_pair(
    IMG_PAIR_DIR,
    windows = window_list,
    save_intermediate_products = save_intermediate_products,
    NP_min = NP_min,
    NP_max = NP_max,
    landmask_shapefile_path = landmask_shapefile_path,
    erode_landmask = None,
    resample = resample,
    resample_interval = resample_interval,
)


## 5. Load resampled RGB and apply landmask

`fully_process_single_image_pair` only writes GeoTIFFs to disk — read back the resampled RGB and resampled landmask, and combine them the same way the STAC/xarray notebook's `rgb_resampled_landmasked` output is built, ready to pass to SAM.

In [ ]:
from osgeo import gdal
import numpy as np

rgb_path = IMG_PAIR_DIR / f"normcovar__RGB__resampled_{resample_interval}_{resample_interval}.tif"
landmask_path = IMG_PAIR_DIR / f"landmask__resampled_{resample_interval}_{resample_interval}.tif"

rgb_resampled = gdal.Open(str(rgb_path)).ReadAsArray().transpose(1, 2, 0)  # (bands, H, W) -> (H, W, 3)
landmask_resampled = gdal.Open(str(landmask_path)).ReadAsArray()  # (H, W), 1 = land, 0 = water

# Match the xarray pipeline's `rgb_resampled_landmasked`: zero out land pixels
sam_input = rgb_resampled.copy()
sam_input[landmask_resampled == 1] = 0
sam_input = sam_input.astype(np.uint8)

## 6. Run SAM Segmentation

The SAM setup/inference logic (imports, weight download, model + mask generator setup, mask generation, and non-overlapping segment ID map construction).

In [ ]:
from fast_ice.sam.model import SAMSegmenter
from fast_ice.sam.predict import annotate_masks, plot_segmentation, plot_segment_id_map
from fast_ice.sam.constants import FAST_MASK_GENERATOR_SETTINGS

# Downloads weights (if needed) and loads the model + mask generator on first use.
# Uses the fast/testing mask generator settings for a quick run; drop mask_generator_settings
# for the default (dense, high-recall) preset, or pass mask_generator_kwargs={...} to override.
segmenter = SAMSegmenter(model_type="vit_h", mask_generator_settings=FAST_MASK_GENERATOR_SETTINGS)
segment_id_map, sam_result = segmenter.predict(sam_input)

annotated = annotate_masks(sam_input, sam_result)
plot_segmentation(sam_input, annotated, title=f"dates: {date1}_{date2}")
plot_segment_id_map(segment_id_map, title=f"Segment ID map — smallest area wins | dates: {date1}_{date2}")

## 7. SVM - Download and Train the SVM if Weights Don't Exist

In [ ]:
from fast_ice.svm.constants import FAST_ICE_CLASS_VALUES
from fast_ice.datasets.svm_training_data import fetch_svm_trainingdata
from fast_ice.svm.train import train
from pathlib import Path

SVM_WEIGHTS = '../fast_ice/svm/weights/2026_08_02_svm_weights.joblib'
SVM_TRAINING_DATA_FOLDER = 'data'

if not Path(SVM_WEIGHTS).exists():
    svm_training_data_dir = fetch_svm_trainingdata(SVM_TRAINING_DATA_FOLDER)
    print(svm_training_data_dir)
    train(svm_training_data_dir, SVM_WEIGHTS)

## 8. Classify SAM Clusters with SVM

In [ ]:
from fast_ice.svm.predict import predict_on_image_and_segments, plot_predictions
pred_map, segment_class = predict_on_image_and_segments(sam_input, segment_id_map, model=SVM_WEIGHTS)
plot_predictions(sam_input, pred_map)

## 9. Convert output to a binary mask and save as Geotiff

In [ ]:
binary_fast_ice = np.where(np.isin(pred_map, FAST_ICE_CLASS_VALUES), 1, 0).astype(np.uint8)

tif_name = IMG_PAIR_DIR / f"{date1}_{date2}_fast_ice.tif"

rgb_ds = gdal.Open(str(rgb_path), gdal.GA_ReadOnly)
driver = gdal.GetDriverByName("GTIFF")
out_ds = driver.Create(str(tif_name), rgb_ds.RasterXSize, rgb_ds.RasterYSize, 1, gdal.GDT_Byte, options=["COMPRESS=LZW"])
out_ds.SetGeoTransform(rgb_ds.GetGeoTransform())
out_ds.SetProjection(rgb_ds.GetProjection())
out_ds.GetRasterBand(1).WriteArray(binary_fast_ice)
out_ds.FlushCache()
out_ds = None
rgb_ds = None

print(f"Saved: {tif_name}")